[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [SQLModel, Deep Dive](https://johnfisher-ai.github.io/Python-Visual-Guides/sqlmodel-deep-dive.html)

# Loading and N+1 &middot; Solutions


One way to do each task. Not the only way. If yours runs and does what was asked, yours is
right too.

The first cell is the notebook's Setup: the classes, `hero_engine`, `counting`, and twenty teams
with two hundred heroes. Run it first. The tasks only read, so they can be run in any order, and the
last cell removes the scratch folder.


In [1]:
import contextlib
import logging
import re
import shutil
import subprocess
import sys
from collections import Counter
from importlib.metadata import PackageNotFoundError, version
from pathlib import Path

try:
    version("sqlmodel")
except PackageNotFoundError:                                        # Colab has no SQLModel: install the pinned version
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "--root-user-action=ignore",
                    "sqlmodel==0.0.42"], check=True)

import sqlmodel
from sqlalchemy import event, func, insert
from sqlalchemy.orm import joinedload, selectinload
from sqlalchemy.orm.exc import DetachedInstanceError
from sqlmodel import Field, Relationship, Session, SQLModel, col, create_engine, select

def message(error):
    """An error's text, without the memory address or the version link that make no two runs agree."""
    text = re.sub(r"0x[0-9a-f]+", "0x...", str(error))
    return "\n".join(line for line in text.splitlines() if "errors.pydantic.dev" not in line).strip()


@contextlib.contextmanager
def counting(engine):
    """The statements an engine sends inside the block, counted by their first word."""
    counted = Counter()

    def count(connection, cursor, statement, parameters, context, executemany):
        counted[statement.split()[0].upper()] += 1

    event.listen(engine, "before_cursor_execute", count)
    try:
        yield counted
    finally:
        event.remove(engine, "before_cursor_execute", count)


class PrintStatements(logging.Handler):
    """Print what an engine logs, leaving out the time: every statement, and the values sent with it."""

    def emit(self, record):
        if record.msg == "[%s] %r":                  # after a statement: how long it took, then its values
            values = repr(record.args[1])
            if values != "()":
                print("    values:", values)
        else:
            for line in record.getMessage().splitlines():
                print("   ", line.rstrip())


sql_log = logging.getLogger("sqlalchemy.engine.Engine")
sql_log.handlers = [PrintStatements()]              # this handler alone, however often the cell runs
sql_log.propagate = False                           # and no handler above it prints the same lines again

def run_python(path):
    """Run a file in a Python of its own and print what it printed."""
    done = subprocess.run([sys.executable, path], capture_output=True, text=True)
    print(done.stdout.strip() or done.stderr.strip().splitlines()[-1])


class Team(SQLModel, table=True):
    id: int | None = Field(default=None, primary_key=True)
    name: str = Field(index=True, max_length=50)
    headquarters: str = Field(max_length=60)

    heroes: list["Hero"] = Relationship(back_populates="team")


class Hero(SQLModel, table=True):
    id: int | None = Field(default=None, primary_key=True)
    name: str = Field(index=True, max_length=50)
    secret_name: str = Field(max_length=60)
    age: int | None = Field(default=None, index=True)
    team_id: int | None = Field(default=None, foreign_key="team.id")

    team: Team | None = Relationship(back_populates="heroes")


def hero_engine(path=None, echo=False):
    """The guide's engine: the database in a file, or with no path one in memory, with foreign keys checked."""
    engine = create_engine("sqlite://" if path is None else f"sqlite:///{path}", echo=echo)

    @event.listens_for(engine, "connect")
    def enforce_foreign_keys(connection, record):
        cursor = connection.cursor()
        cursor.execute("PRAGMA foreign_keys=ON")
        cursor.close()

    return engine


PREFIXES = ["Iron", "Silver", "Night", "Storm", "Ghost", "Solar", "Crimson", "Jade", "Cobalt", "Ember"]
BEASTS = ["Fox", "Hawk", "Wolf", "Lark", "Viper", "Moth", "Crane", "Otter", "Falcon", "Bear",
          "Lynx", "Heron", "Stag", "Raven", "Mole", "Owl", "Pike", "Shrike", "Boar", "Kite"]

BIG_TEAMS = [f"{prefix} Squad" for prefix in PREFIXES] + [f"{beast} Watch" for beast in BEASTS[:10]]
BIG_HEROES = [f"{prefix}-{beast}" for prefix in PREFIXES for beast in BEASTS]


def build_many(engine):
    """Twenty teams and two hundred heroes, ten to a team, from the lists above and no random."""
    SQLModel.metadata.create_all(engine)
    with engine.begin() as connection:
        connection.execute(insert(Team), [{"name": name, "headquarters": f"{name} House"}
                                          for name in BIG_TEAMS])
        connection.execute(insert(Hero), [{"name": name, "secret_name": f"Recruit {number:03d}",
                                           "age": 20 + number % 40, "team_id": (number % 20) + 1}
                                          for number, name in enumerate(BIG_HEROES)])

shutil.rmtree("scratch", ignore_errors=True)                        # a rerun starts from the same rows
Path("scratch").mkdir()
engine = hero_engine("scratch/heroes.db")
build_many(engine)

with Session(engine) as session:
    print("sqlmodel", sqlmodel.__version__, "|", len(session.exec(select(Hero)).all()), "heroes in",
          len(session.exec(select(Team)).all()), "teams")


sqlmodel 0.0.42 | 200 heroes in 20 teams


**1.** A loop, counted.


In [2]:
with Session(engine) as session, counting(engine) as sent:
    older = session.exec(select(Hero).where(Hero.age > 50)).all()
    names = {hero.team.name for hero in older}
print(len(older), "heroes on", len(names), "teams |", dict(sent))
# One SELECT for the heroes, then one for each distinct team they are on. The count follows the
# number of teams, not the number of heroes.


45 heroes on 9 teams | {'SELECT': 10}


The heroes over 50 are spread across every team, so the loop asked for all twenty.


**2.** The same, loaded with the heroes.


In [3]:
with Session(engine) as session, counting(engine) as one_at_a_time:
    for hero in session.exec(select(Hero).where(Hero.age > 50)).all():
        hero.team.name

with Session(engine) as session, counting(engine) as together:
    for hero in session.exec(select(Hero).where(Hero.age > 50).options(joinedload(Hero.team))).all():
        hero.team.name

print("one at a time:", dict(one_at_a_time))
print("joinedload   :", dict(together))


one at a time: {'SELECT': 10}
joinedload   : {'SELECT': 1}


Twenty-one statements against one. `joinedload` is the right choice here because a hero has one
team: the join adds columns rather than rows.


**3.** The ten oldest, with their teams.


In [4]:
with Session(engine) as session, counting(engine) as sent:
    oldest = session.exec(select(Hero).options(joinedload(Hero.team))
                          .order_by(col(Hero.age).desc(), Hero.name).limit(10)).all()
    rows = [(hero.name, hero.age, hero.team.name) for hero in oldest]

for row in rows[:3]:
    print(" ", row)
print("statements:", dict(sent))


  ('Ember-Kite', 59, 'Bear Watch')
  ('Jade-Kite', 59, 'Bear Watch')
  ('Silver-Kite', 59, 'Bear Watch')
statements: {'SELECT': 1}


One statement for ten heroes and their teams. Without the option it would have been one plus the
number of distinct teams among those ten.


**4.** A page of teams, both ways.


In [5]:
with Session(engine) as session, counting(engine) as plain:
    [(team.name, len(team.heroes)) for team in session.exec(select(Team).limit(3)).all()]

with Session(engine) as session, counting(engine) as loaded:
    [(team.name, len(team.heroes))
     for team in session.exec(select(Team).options(selectinload(Team.heroes)).limit(3)).all()]

print("without an option:", dict(plain))
print("with selectinload:", dict(loaded))


without an option: {'SELECT': 4}
with selectinload: {'SELECT': 2}


Four against two, and the four would have been twenty-one for the whole list.


**5.** Teams that outlive their session.


In [6]:
with Session(engine) as session:
    teams = session.exec(select(Team).options(selectinload(Team.heroes)).limit(2)).all()

for team in teams:
    print(f"  {team.name}: {len(team.heroes)} heroes, first is {sorted(h.name for h in team.heroes)[0]}")


  Iron Squad: 10 heroes, first is Cobalt-Fox
  Silver Squad: 10 heroes, first is Cobalt-Hawk


Everything the loop reads was fetched while the session was open, so the block can end before any
of it is used.


**6.** The same counts, with no relationship at all.


In [7]:
def team_sizes(session):
    """Every team's name with how many heroes it has, in one statement."""
    counted = (select(Team.name, func.count(Hero.id))
               .join(Hero, isouter=True)
               .group_by(Team.name))
    return {name: heroes for name, heroes in session.exec(counted)}


with Session(engine) as session, counting(engine) as sent:
    sizes = team_sizes(session)
print("statements:", dict(sent), "| teams:", len(sizes))
print("three of them:", dict(sorted(sizes.items())[:3]))


statements: {'SELECT': 1} | teams: 20
three of them: {'Bear Watch': 10, 'Cobalt Squad': 10, 'Crane Watch': 10}


One statement, and no objects loaded at all. When a page needs a number rather than the rows, the
database is the place to work it out, and no loading strategy is involved.

Last, the engine lets go of the file, and this cell removes the scratch folder:


In [8]:
engine.dispose()
shutil.rmtree("scratch")

print("scratch still there:", Path("scratch").exists())


scratch still there: False


---

&#8592; **Back to:** [Loading and N+1](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/sqlmodel-deep-dive/10-loading-and-n-plus-one.ipynb)  &nbsp;&middot;&nbsp;  [SQLModel, Deep Dive Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/sqlmodel-deep-dive.html)
